In [60]:
import pandas as pd
import numpy as np
import os
import importlib
import EDA as eda 
import Feature_eng as feng
importlib.reload(eda)
importlib.reload(feng)



<module 'Feature_eng' from 'c:\\Users\\kaspe\\OneDrive\\Documents\\USYD-Kasper\\Year 3\\QBUS3820\\Assignment\\Code\\QBUS3820-Group-Assignment\\Feature_eng.py'>

In [3]:
transactions, demographics, products, campaigns, campaign_descriptions, promotions = eda.retrieve_data()

## Demographics

In [61]:
demographics_train, demographics_valid, demographics_test = eda.clean_demographics(eda.demographics)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1469307 entries, 0 to 1469306
Data columns (total 11 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   household_id           1469307 non-null  int64         
 1   store_id               1469307 non-null  int64         
 2   basket_id              1469307 non-null  int64         
 3   product_id             1469307 non-null  int64         
 4   quantity               1469307 non-null  int64         
 5   sales_value            1469307 non-null  float64       
 6   retail_disc            1469307 non-null  float64       
 7   coupon_disc            1469307 non-null  float64       
 8   coupon_match_disc      1469307 non-null  float64       
 9   week                   1469307 non-null  int64         
 10  transaction_timestamp  1469307 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(4), int64(6)
memory usage: 123.3 MB
Missing values in trans

### Demographics Data Cleaning

Considering the amount of missing values in home_ownership and marital status, and the fact that they most likely don't have much informative data on customer churn, we remove these columns from the dataset. 

In [71]:
print(demographics_train.head())
print(demographics_valid.head())
print(demographics_test.head())
print(f"Training datapoints: {demographics_train.shape[0]}")
print(f"Validation datapoints: {demographics_valid.shape[0]}")
print(f"Test datapoints: {demographics_test.shape[0]}")

   household_id    age     income household_size    household_comp kids_count
0             1    65+     35-49K              2  2 Adults No Kids          0
1          1001  45-54     50-74K              1   1 Adult No Kids          0
2          1003  35-44     25-34K              1   1 Adult No Kids          0
3          1004  25-34     15-24K              1   1 Adult No Kids          0
4           101  45-54  Under 15K              4     2 Adults Kids          2
    household_id    age     income household_size    household_comp kids_count
11          1024  25-34  Under 15K              4     2 Adults Kids          2
26          1070  35-44     50-74K              2  2 Adults No Kids          0
41           113  35-44   125-149K              4     2 Adults Kids          2
44          1135  45-54     50-74K              2      1 Adult Kids          1
45          1137    65+     35-49K              1   1 Adult No Kids          0
    household_id    age    income household_size    househ

### Demographics Visualisation

## TRANSACTIONS

In [14]:
transactions.info()
continous_transactions = ["sales_value","retail_disc","coupon_disc","coupon_match_disc"]
discrete_transactions = ["quantity"]
nominal_categorical_transactions = ["household_id", "store_id", "basket_id", "product_id"]
ordinal_categorical_transactions = ["week"]
duplicates= transactions.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

for col in ordinal_categorical_transactions:
    print(f"\n{col.upper()} - Unique Values:")
    print(transactions[col].unique())

print("Week number- Unique Values:")
print(transactions['week'].unique())


# Split transactions data into train, valid, and test sets by household_id
# This ensures no household appears in multiple datasets for better generalization

unique_households = transactions['household_id'].unique()
np.random.seed(42)  # For reproducibility
np.random.shuffle(unique_households)

# First split: 70% train, 30% temp (valid + test combined)
train_size = int(0.7 * len(unique_households))
train_households = unique_households[:train_size]

# Second split: Split the remaining 30% into 50% valid, 50% test
temp_households = unique_households[train_size:]
temp_size = len(temp_households)
valid_size = int(0.5 * temp_size)
valid_households = temp_households[:valid_size]
test_households = temp_households[valid_size:]

# Create transaction datasets based on household splits
transactions_train = transactions[transactions['household_id'].isin(train_households)].copy()
transactions_valid = transactions[transactions['household_id'].isin(valid_households)].copy()
transactions_test = transactions[transactions['household_id'].isin(test_households)].copy()

print(f"\nTransactions Split Summary:")
print(f"Total unique households: {len(unique_households)}")
print(f"Train households: {len(train_households)} ({100*len(train_households)/len(unique_households):.1f}%)")
print(f"Valid households: {len(valid_households)} ({100*len(valid_households)/len(unique_households):.1f}%)")
print(f"Test households: {len(test_households)} ({100*len(test_households)/len(unique_households):.1f}%)")
print(f"\nTransaction counts:")
print(f"Train transactions: {len(transactions_train)}")
print(f"Valid transactions: {len(transactions_valid)}")
print(f"Test transactions: {len(transactions_test)}")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1469307 entries, 0 to 1469306
Data columns (total 11 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   household_id           1469307 non-null  int64         
 1   store_id               1469307 non-null  int64         
 2   basket_id              1469307 non-null  int64         
 3   product_id             1469307 non-null  int64         
 4   quantity               1469307 non-null  int64         
 5   sales_value            1469307 non-null  float64       
 6   retail_disc            1469307 non-null  float64       
 7   coupon_disc            1469307 non-null  float64       
 8   coupon_match_disc      1469307 non-null  float64       
 9   week                   1469307 non-null  int64         
 10  transaction_timestamp  1469307 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(4), int64(6)
memory usage: 123.3 MB
Number of duplicate row

In [59]:
churn = feng.churn(transactions, threshold_days=21)
print(churn.head())
print(churn.value_counts())

household_id
1    0
2    1
3    0
4    1
5    1
Name: churn, dtype: int64
churn
0    1897
1     572
Name: count, dtype: int64
